In [ ]:
# Install dependencies (only needed on first run)
%pip install git+https://github.com/huggingface/transformers.git
%pip install accelerate
%pip install paddlepaddle-gpu
%pip install "paddleocr>=2.0.1"

In [ ]:
# Fix encoding issue on Google Colab (skip on local machine)
import locale
locale.getpreferredencoding = lambda do_setlocale=True: "UTF-8"

In [ ]:
import json
import cv2
import torch
from paddleocr import PaddleOCR
from transformers import pipeline
from google.colab.patches import cv2_imshow

In [ ]:
# Load OCR model (lang can be 'en', 'ch', 'fr', 'german', 'korean', 'japan')
reader = PaddleOCR(use_angle_cls=True, lang='en', use_space_char=True, show_log=False, enable_mkldnn=True)

In [ ]:
# Show the bill image
image_file = '/content/test_2.jpg'
image = cv2.imread(image_file, cv2.IMREAD_UNCHANGED)
cv2_imshow(image)

In [ ]:
# Run OCR on the image
ocr_result = reader.ocr(image_file, cls=True)

In [ ]:
# Join all detected lines into one string
raw_text = " ".join(line[1][0] for line in ocr_result[0])
raw_text

In [ ]:
# Load Zephyr-7B for parsing the OCR text
llm = pipeline("text-generation", model="HuggingFaceH4/zephyr-7b-alpha", torch_dtype=torch.bfloat16, device_map="auto")

In [ ]:
# Prompt for the LLM (last message must be from "user")
system_msg = "You are a JSON converter which receives raw boarding pass OCR information as a string and returns a structured JSON output by organising the information in the string."
user_msg = "Extract the name of the passenger, name of the airline, Flight number, City of Departure, City of Arrival, Date of Departure from this OCR data: {}"
gen_args = dict(max_new_tokens=1000, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)

def build_prompt(text):
    chat = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg.format(text)},
    ]
    return llm.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

In [ ]:
# Test with a sample boarding pass text
test_text = "BOARDING PASS BOARDING PASS JOHN SMITH PASSENGER PASSENGER SEC JOHN SMITH NYC 16C FROM TO FLYING FROM SEAT NYC LA LA B27 TO GATE FLIGHT NUMBER BH01122 FLIGHT BH01122 GATE B27 DEPARTURE DATE 25MAY DATE 25MAY DEPARTURE TIME 20:10 SEAT 16C GATE CLOSES 15 MINUTES BEFORE DEPARTURE C"
prompt_text = build_prompt(test_text)
print(prompt_text)

In [ ]:
response = llm(prompt_text, **gen_args)
print(response[0]["generated_text"])

In [ ]:
# All in one: image -> OCR -> LLM -> JSON file
def extract_bill(image_file, out_file='result.json'):
    ocr_result = reader.ocr(image_file, cls=True)
    raw_text = " ".join(line[1][0] for line in ocr_result[0])

    response = llm(build_prompt(raw_text), return_full_text=False, **gen_args)
    reply = response[0]["generated_text"]
    print(reply)

    # Keep only the JSON part of the reply
    start, end = reply.find("{"), reply.rfind("}") + 1
    try:
        data = json.loads(reply[start:end])
    except json.JSONDecodeError as e:
        print(f"Could not parse JSON: {e}")
        return None

    with open(out_file, "w") as file:
        json.dump(data, file, indent=4)

    print(f"Saved to {out_file}")
    return data

In [ ]:
extract_bill('/content/test_2.jpg')